# Generation evaluation, step by step

This notebook opens the Phase 4 report and **reads it the way an engineer would**: one small
step at a time, with a chart for every question we ask.

Every step has the same shape:

- **Why** — what we care about, and what could go wrong if we didn't look.
- **What we do** — the one thing the code cell does.
- **What to notice** — how to read the output or chart.

**Before you start**, produce the two report files (they are git-ignored, so each machine
makes its own):

```bash
uv run python scripts/run_evaluation.py              # -> reports/retrieval_report.json  (seconds)
uv run python scripts/run_generation_evaluation.py   # -> reports/generation_report.json (~1 hour)
```

Then pick the project's `.venv` kernel and run the cells from top to bottom
(needs `uv sync --all-extras --all-groups`).

| Part | Steps | Question it answers |
|---|---|---|
| A · Load | 1–3 | What is inside the report? |
| B · The headline | 4–6 | Did we pass, and is the average telling the truth? |
| C · Look closer | 7–10 | Which answers are weak, and why? |
| D · Connect the phases | 11–12 | Do retrieval misses cause bad answers? Where does time go? |
| E · See the judge work | 13 | What does the judge actually do with one answer? |
| F · Compare runs | 14–15 | Did a change make things better or worse? |

## Part A — Load the results

### Step 1 · Set up the notebook

**Why:** the scripts read `data/` and `reports/` relative to the repository root. If the
notebook runs from `notebooks/`, those paths would point to the wrong place.

**What we do:** move to the repository root, import pandas (tables) and matplotlib (charts),
and fix a few colours so every chart uses the same meaning: green = pass, red = fail,
grey dashed line = pass mark.

In [ ]:
import json
import os
import warnings
from pathlib import Path

# Keep the output readable: hide download progress bars and library chatter.
os.environ.setdefault("TQDM_DISABLE", "1")
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("HF_HUB_VERBOSITY", "error")
os.environ.setdefault("TRANSFORMERS_VERBOSITY", "error")
warnings.filterwarnings("ignore", message=".*IProgress not found.*")
warnings.filterwarnings("ignore", message=".*unauthenticated requests to the HF Hub.*")

import matplotlib.pyplot as plt  # noqa: E402 - after the environment settings above
import pandas as pd  # noqa: E402

# Walk up from the current folder until we find pyproject.toml: that is the repository root.
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(root)

from rag_eval_platform.config.settings import get_settings  # noqa: E402

settings = get_settings()
pd.set_option("display.max_colwidth", 110)
plt.rcParams.update({"figure.figsize": (8, 4), "axes.spines.top": False,
                     "axes.spines.right": False})  # fmt: skip

PASS, FAIL, NEUTRAL, MARK = "#16a34a", "#dc2626", "#2563eb", "#111827"
print("Working in:", Path.cwd())

### Step 2 · Open the report file

**Why:** before trusting any number, know **what produced it**. A score only means something
together with the model that wrote the answers, the prompt version, and the judge that graded
them. Change any of these and the numbers are no longer comparable.

**What we do:** load the JSON file and print its "label": models, prompt versions, pass marks.

In [ ]:
report_path = Path("reports/generation_report.json")
if not report_path.exists():
    # The report is written only when a full evaluation run finishes (about 1 hour locally).
    raise SystemExit(
        f"{report_path} does not exist yet. Create it first:\n"
        "  uv run python scripts/run_generation_evaluation.py            # all 30 questions\n"
        "  uv run python scripts/run_generation_evaluation.py --limit 3  # quick, 3 questions"
    )
report = json.loads(report_path.read_text(encoding="utf-8"))

print("Top-level keys:", list(report))
print()
print("Answers written by :", report["generator_model"], "| prompt", report["prompt_version"])
print("Answers graded by  :", report["judge_model"],
      "| citation prompt", report["citation_prompt_version"])  # fmt: skip
print("Context metrics    :", "yes (--full)" if report["full"] else "no")
print("Pass marks         :", report["thresholds"])
print("Passed?            :", report["passed"], report["failures"])

**What to notice:** the generator (`qwen3:8b`) and the judge (`gemma3:12b`) are **different
models**. That is on purpose: a model grading its own answers tends to be too kind
(*self-preference bias*).

### Step 3 · One row per question

**Why:** the report stores each question as a nested dictionary. A **table** (a pandas
`DataFrame`) lets us sort, filter, group and plot with one line each.

**What we do:** pull the fields we care about out of each example into flat columns.

In [ ]:
rows = []
for ex in report["examples"]:
    rows.append(
        {
            "id": ex["example_id"],
            "type": ex["query_type"],
            "faithfulness": ex["scores"]["faithfulness"],
            "relevance": ex["scores"]["answer_relevance"],
            "citation_validity": ex["citation_validity"],
            "refusal": ex["refusal"],
            "invalid_citations": len(ex["invalid_citations"]),
            "unsupported_citations": len(ex["unsupported_citations"]),
            "seconds": ex["generation_ms"] / 1000,
            "question": ex["question"],
            "answer": ex["answer"],
        }
    )
gen = pd.DataFrame(rows)

print(len(gen), "questions")
gen.head()

**What to notice:** each column is one **measurement** of one answer:

| Column | Meaning | Good value |
|---|---|---|
| `faithfulness` | share of the answer's claims the retrieved chunks support | 1.0 |
| `relevance` | how directly the answer addresses the question | close to 1.0 |
| `citation_validity` | share of `[n]` citations whose chunk really supports the sentence | 1.0 |
| `refusal` | did the model say "I don't know…"? (every golden question is answerable) | False |
| `seconds` | time the generator took to write the answer | small |

An empty cell (`NaN` / `None`) means **not scored**, for example citation validity for an
answer that cites nothing.

## Part B — The headline

### Step 4 · Overall scores against the pass marks

**Why:** this is the question a pull request asks: *is quality still above the bar?*
Faithfulness and answer relevance have pass marks (0.85 and 0.80); the other numbers are
tracked without a gate.

**What we do:** draw one bar per metric, colour it by pass/fail, and draw the pass mark as a
dashed line.

In [ ]:
overall = report["overall"]
marks = {"faithfulness": report["thresholds"]["min_faithfulness"],
         "answer_relevance": report["thresholds"]["min_answer_relevance"]}  # fmt: skip
metrics = ["faithfulness", "answer_relevance", "citation_validity"]
values = [overall[m] or 0.0 for m in metrics]
colours = [
    NEUTRAL if m not in marks else (PASS if overall[m] >= marks[m] else FAIL) for m in metrics
]

fig, ax = plt.subplots()
bars = ax.bar(metrics, values, color=colours)
ax.bar_label(bars, fmt="%.3f")
for i, m in enumerate(metrics):
    if m in marks:
        ax.hlines(marks[m], i - 0.4, i + 0.4, colors=MARK, linestyles="dashed", linewidth=2)
ax.set_ylim(0, 1.1)
ax.set_title("Overall scores (dashed = pass mark, blue = no pass mark)")
plt.show()

**What to notice:** a bar above its dashed line passes. Citation validity has no pass mark
yet: we first need a baseline to know what "normal" looks like before we set a bar.

### Step 5 · Check the report by hand

**Why:** *trust, but verify.* If our own arithmetic gives the same numbers as the report, we
understand exactly what each number means. If it doesn't, one of us has a bug.

**What we do:** recompute the averages and the two rates from the table.

In [ ]:
judged = gen[~gen.refusal]  # refusals are not sent to the judge

by_hand = {
    # .mean() skips empty cells, like the report does
    "faithfulness": judged.faithfulness.mean(),
    # refusals count as relevance 0, so average over ALL answers
    "answer_relevance": gen.relevance.mean(),
    "citation_validity": gen.citation_validity.mean(),
    # hallucination rate = share of judged answers with ANY unsupported claim (faithfulness < 1)
    "hallucination_rate": (judged.faithfulness.dropna() < 1.0).mean(),
    "refusal_rate": gen.refusal.mean(),
    "invalid_citation_rate": (gen.invalid_citations > 0).mean(),
}
check = pd.DataFrame({"by hand": by_hand, "report": {k: overall[k] for k in by_hand}})
check["same?"] = (check["by hand"] - check["report"]).abs() < 1e-9
check.round(3)

**What to notice:** every row should say `True`. Two definitions worth remembering:

- **Faithfulness** (an average) says *how much* of an answer is supported, on average.
- **Hallucination rate** (a share) says *how often* an answer contains **anything**
  unsupported. An average faithfulness of 0.95 can still hide a hallucination rate of 30%.

### Step 6 · Averages hide things: look at the spread

**Why:** an average of 0.9 could mean "every answer is 0.9" or "most are 1.0 and a few are
terrible". Those need very different fixes. A **histogram** shows the spread.

**What we do:** plot how many answers fall into each faithfulness band.

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
bins = [i / 10 for i in range(11)]

left.hist(judged.faithfulness.dropna(), bins=bins, color=NEUTRAL, edgecolor="white")
left.axvline(marks["faithfulness"], color=MARK, linestyle="dashed")
left.set(title="Faithfulness per answer", xlabel="score", ylabel="number of answers")

right.hist(gen.relevance.dropna(), bins=bins, color=NEUTRAL, edgecolor="white")
right.axvline(marks["answer_relevance"], color=MARK, linestyle="dashed")
right.set(title="Answer relevance per answer", xlabel="score")
plt.show()

print("Answers below the faithfulness pass mark:",
      int((judged.faithfulness < marks["faithfulness"]).sum()), "of", len(judged))  # fmt: skip

**What to notice:** a tall bar at 1.0 with a few answers to the left is the typical shape:
most answers are fully grounded, and the average is pulled down by a handful. Those few are
exactly what Part C looks at.

## Part C — Look closer

### Step 7 · Break it down by question type

**Why:** the golden set has three kinds of question: `short` (one fact), `paraphrase` (same
meaning, different words) and `multi_hop` (needs two documents). In Phase 2, retrieval was
weakest on multi-hop. Does that weakness carry into the answers?

**What we do:** average each metric per question type and draw grouped bars.

In [ ]:
per_type = gen.groupby("type")[["faithfulness", "relevance", "citation_validity"]].mean()
per_type["n"] = gen.groupby("type").size()
display(per_type.round(3))

ax = per_type[["faithfulness", "relevance", "citation_validity"]].plot.bar(rot=0, width=0.8)
ax.axhline(marks["faithfulness"], color=MARK, linestyle="dashed", linewidth=1)
ax.set(ylim=(0, 1.1), title="Scores by question type (dashed = faithfulness pass mark)",
       xlabel="")  # fmt: skip
ax.legend(bbox_to_anchor=(1, 1), loc="upper left")
plt.show()

**What to notice:** compare the bars **between** groups, and check `n`: a group of 6
questions moves a lot when one answer changes, so treat small groups with care.

### Step 8 · Two scores at once: faithfulness vs relevance

**Why:** the two metrics catch **different** failures, and a scatter plot shows both at once:

| Where the dot is | What it means | Typical cause |
|---|---|---|
| top right | grounded **and** on topic | 👍 |
| low faithfulness | makes a claim the chunks don't support | hallucination, or a paraphrase the judge can't match |
| low relevance | true, but doesn't answer the question | vague or evasive answer, or a refusal |

**What we do:** plot every answer as a dot, coloured by question type, with the pass marks as
lines that split the chart into four boxes.

In [ ]:
colours_by_type = {"short": "#2563eb", "paraphrase": "#16a34a", "multi_hop": "#d97706"}

fig, ax = plt.subplots(figsize=(7, 6))
for qtype, group in gen.groupby("type"):
    ax.scatter(group.relevance, group.faithfulness.fillna(0), label=qtype, s=60,
               alpha=0.8, color=colours_by_type.get(qtype))  # fmt: skip
ax.axvline(marks["answer_relevance"], color=MARK, linestyle="dashed")
ax.axhline(marks["faithfulness"], color=MARK, linestyle="dashed")

# Name only the dots below the faithfulness line (few, so the labels stay readable).
unfaithful = gen[gen.faithfulness.fillna(0) < marks["faithfulness"]]
for _, row in unfaithful.iterrows():
    y = 0.0 if pd.isna(row.faithfulness) else row.faithfulness  # refusals have no score
    ax.annotate(row["id"], (row.relevance, y), fontsize=8,
                xytext=(4, 4), textcoords="offset points")  # fmt: skip
ax.set(xlim=(-0.05, 1.05), ylim=(-0.05, 1.05), xlabel="answer relevance",
       ylabel="faithfulness", title="Every answer: grounded? on topic?")  # fmt: skip
ax.legend(loc="lower left")
plt.show()

# The dots left of the relevance line are many and crowd the top row, so list them instead.
off_topic = gen[gen.relevance < marks["answer_relevance"]].sort_values("relevance")
print("Below the relevance pass mark:", len(off_topic))
off_topic[["id", "type", "relevance", "faithfulness", "question"]].round(2)

### Step 9 · Read the weakest answers

**Why:** a chart tells you **where** to look; only reading tells you **why**. An engineer
always reads the worst cases before changing anything.

**What we do:** sort by faithfulness and print the question and answer of the lowest ones.

In [ ]:
# Unscored (refusals) first: an unanswered golden question is the worst case of all.
worst = gen.sort_values(["faithfulness", "relevance"], na_position="first").head(5)
for _, row in worst.iterrows():
    print(f"{row['id']} ({row['type']})  faithfulness={row.faithfulness}  "
          f"relevance={row.relevance:.2f}")  # fmt: skip
    print("  Q:", row.question)
    print("  A:", row.answer)
    print()

**What to notice:** for each weak answer, ask *which layer* is to blame:

1. **Retrieval** — the right chunk wasn't retrieved (check it in Step 11).
2. **Generation** — the chunk was there, but the model added something or overstated it.
3. **The judge** — the answer is fine, but the judge misread a paraphrase. LLM judges are
   models too; that is why the methodology says *spot-check verdicts by hand*.

### Step 10 · Citations: do the footnotes point to the right chunk?

**Why:** a citation is a promise: "this sentence comes from chunk *n*". Phase 3 could only
check that chunk *n* **exists**. The judge now checks it **supports** the sentence.

**What we do:** count citations per answer, then list every one the judge found unsupported.

In [ ]:
cited = gen.dropna(subset=["citation_validity"]).sort_values("citation_validity")

fig, ax = plt.subplots(figsize=(10, 4))
# A score of 0 would draw a bar of height 0 (invisible), so give every bar a small minimum
# height and write the real value on the red ones.
heights = cited.citation_validity.clip(lower=0.02)
bars = ax.bar(cited["id"], heights,
              color=[PASS if v == 1 else FAIL for v in cited.citation_validity])  # fmt: skip
ax.bar_label(bars, labels=[f"{v:.2f}" if v < 1 else "" for v in cited.citation_validity],
             fontsize=8)  # fmt: skip
ax.set(ylim=(0, 1.05), title="Citation validity per answer (red = at least one unsupported)",
       ylabel="share of supported citations")  # fmt: skip
ax.tick_params(axis="x", rotation=75, labelsize=8)
plt.show()

for ex in report["examples"]:
    for number, sentence in ex["unsupported_citations"]:
        print(f"{ex['example_id']} [{number}]: {sentence}")

**What to notice:** an unsupported citation is not always a wrong fact. Often the sentence
is true and supported by a **different** chunk (the model cited `[4]` when the fact is in
`[1]`). Faithfulness would still be 1.0 there: the claim is backed, just by the wrong
footnote. That is why citation validity is a **separate** number.

**A limitation you can see above:** one sentence reads *"such as,,,"*. That answer was
*explaining* citations, and its text contained example markers like `[1], [2], [3]`. The
checker cannot tell an example from a real citation, so it treated them as citations and
removed them from the sentence. Every automatic metric has blind spots like this, which is
another reason to read the flagged cases yourself.

## Part D — Connect the phases

### Step 11 · Do retrieval misses cause bad answers?

**Why:** RAG is a chain: *retrieve → generate*. If the right document never reached the
model, even a perfect writer can't answer faithfully. Joining the Phase 2 retrieval report
with the Phase 4 generation report shows whether weak answers come from weak retrieval.

**What we do:** load the retrieval report, keep recall per question, and **merge** the two
tables on the question id.

In [ ]:
retrieval = json.loads(Path("reports/retrieval_report.json").read_text(encoding="utf-8"))
ret = pd.DataFrame(
    {"id": ex["example_id"], "recall": ex["scores"]["recall"], "mrr": ex["scores"]["mrr"]}
    for ex in retrieval["examples"]
)
both = gen.merge(ret, on="id", how="inner")  # keep questions present in both reports
print(len(both), "questions in both reports")

both["retrieval"] = both.recall.map(lambda r: "found every relevant doc" if r == 1
                                    else "missed a relevant doc")  # fmt: skip
summary = both.groupby("retrieval")[["faithfulness", "relevance", "citation_validity"]].mean()
summary["n"] = both.groupby("retrieval").size()
summary.round(3)

In [ ]:
ax = summary[["faithfulness", "relevance", "citation_validity"]].plot.bar(rot=0, width=0.8)
ax.set(ylim=(0, 1.1), title="Answer quality when retrieval hit vs missed", xlabel="")
ax.legend(bbox_to_anchor=(1, 1), loc="upper left")
plt.show()

**What to notice:** if the "missed" group scores clearly lower, the fix belongs in
**retrieval** (re-ranking, chunk size, top-k), not in the prompt. If both groups look the
same, retrieval misses are not what's hurting answers.

Careful with a small "missed" group: a handful of questions is a hint, not proof.

### Step 12 · Where does the time go?

**Why:** quality is not the only cost. A user waits for every answer, and a slow model makes
evaluation slow too.

**What we do:** plot how long the generator took for each answer, per question type.

In [ ]:
ax = gen.boxplot(column="seconds", by="type", grid=False)
ax.set(title="Generation time per answer", xlabel="", ylabel="seconds")
plt.suptitle("")
plt.show()

print(gen.seconds.describe().round(2))

**What to notice:** the box is the middle half of the answers, the line inside is the
median. Dots far above are slow outliers. The slowest answer is usually the **first
question** of the run: the model is still warming up in memory (a *cold start*). The **judge** time (1–2 minutes per question)
is not in this chart: it only matters for running the evaluation, not for users.

## Part E — See the judge work

### Step 13 · Grade one answer live, and a broken copy of it

**Why:** a judge is only useful if it tells good from bad. The simplest test: take a real
answer, make a **broken copy** with an invented claim, and check the judge scores the copy
lower. This is the same sanity check we ran before trusting it.

**What we do:** build a `JudgeSample` (question, answer, retrieved chunks, reference) and call
the real judge. Needs Ollama running (`brew services start ollama`) with `qwen3:8b` and
`gemma3:12b` pulled; the first cell checks this for you. Takes about 1–2 minutes.

In [ ]:
import urllib.request

from rag_eval_platform.evaluation.golden_dataset import load_golden_dataset
from rag_eval_platform.evaluation.judge import JudgeSample, RagasJudge
from rag_eval_platform.ingestion.embedding import create_embedder
from rag_eval_platform.pipeline import create_pipeline

# 0. Check the local model server first, so a missing service gives a clear message.
tags_url = settings.ollama_base_url.removesuffix("/v1") + "/api/tags"
try:
    with urllib.request.urlopen(tags_url, timeout=3) as response:  # noqa: S310 - local URL
        pulled = {m["name"] for m in json.load(response)["models"]}
except OSError:
    raise SystemExit("Ollama is not running. Start it with: brew services start ollama") from None
missing = {settings.llm_model, settings.judge_model} - pulled
if missing:
    raise SystemExit(f"Pull the missing models first: ollama pull {' '.join(sorted(missing))}")
print("Ollama is running with", settings.llm_model, "and", settings.judge_model)

golden = {ex.id: ex for ex in load_golden_dataset(settings.golden_dataset_path)}
example = golden["overview-001"]

# 1. Get a fresh answer and the chunks it was based on.
answer = create_pipeline(settings).ask(example.question)
contexts = tuple(source.chunk.text for source in answer.sources)
print("Q:", example.question)
print("A:", answer.text)

In [ ]:
judge = RagasJudge.from_settings(settings, create_embedder(settings))

good = JudgeSample(example.question, answer.text, contexts, example.expected_answer)
broken = JudgeSample(
    example.question,
    answer.text + " RAG was invented in 1995 by NASA to guide rockets [1].",  # made up
    contexts,
    example.expected_answer,
)

for name, sample in [("real answer", good), ("broken copy", broken)]:
    scores = judge.score(sample)
    print(f"{name:12} faithfulness={scores.faithfulness:.2f}  "
          f"relevance={scores.answer_relevance:.2f}")  # fmt: skip

**What to notice:** the broken copy's faithfulness should drop, because one of its claims
isn't in any chunk. Relevance may barely move: the answer is still *about* the question. That
is the whole point of having two metrics.

Now look at the exact prompt the **citation** checker sends. No model is needed to build it:

In [ ]:
from rag_eval_platform.evaluation.citation_validity import (
    build_citation_messages,
    extract_cited_claims,
)

claims = extract_cited_claims(answer.text, len(answer.sources))
pairs = [(c.sentence, answer.sources[n - 1].chunk.text[:200] + " ...")
         for c in claims for n in c.numbers]  # fmt: skip
system, user = build_citation_messages(pairs)
print("SYSTEM:\n" + system.content + "\n")
print("USER:\n" + user.content)

## Part F — Compare runs

### Step 14 · A fair comparison needs the same questions

**Why:** every change to chunking, top-k, re-ranking or the prompt must be judged **before vs
after** (the project rule for pull requests). Comparing a 10-question run with a 30-question
run would be unfair: different questions, different difficulty.

**What we do:** load an experiment report, keep only the questions both runs have, and
compare metric by metric.

Create an experiment report first, for example with re-ranking on (about 12 minutes for 10
questions):

```bash
RAG_RERANK=true uv run python scripts/run_generation_evaluation.py --limit 10 \
    --report reports/generation_report_rerank.json
```

In [ ]:
def load_run(path: str) -> pd.DataFrame:
    data = json.loads(Path(path).read_text(encoding="utf-8"))
    return pd.DataFrame(
        {
            "id": ex["example_id"],
            "faithfulness": ex["scores"]["faithfulness"],
            "relevance": ex["scores"]["answer_relevance"],
            "citation_validity": ex["citation_validity"],
        }
        for ex in data["examples"]
    ).set_index("id")


experiment_path = Path("reports/generation_report_rerank.json")
if experiment_path.exists():
    before = load_run("reports/generation_report.json")
    after = load_run(str(experiment_path))
    shared = before.index.intersection(after.index)  # the fair part: same questions only
    before, after = before.loc[shared], after.loc[shared]
    comparison = pd.DataFrame({"before": before.mean(), "after": after.mean()})
    comparison["change"] = comparison["after"] - comparison["before"]
    print(len(shared), "questions in both runs")
    display(comparison.round(3))
else:
    print("No experiment report yet: run the command above, then run this cell again.")

In [ ]:
if experiment_path.exists():
    ax = comparison[["before", "after"]].plot.bar(rot=0, width=0.8,
                                                 color=[MARK, NEUTRAL])  # fmt: skip
    ax.set(ylim=(0, 1.1), title=f"Before vs after on the same {len(shared)} questions")
    ax.legend(bbox_to_anchor=(1, 1), loc="upper left")
    plt.show()

### Step 15 · Which questions changed?

**Why:** an average can stay flat while some questions get better and others get worse (we saw
exactly this with re-ranking in Phase 2). Looking per question shows the **trade-off**.

**What we do:** plot each question's faithfulness before (black) and after (blue), joined by
a line, sorted by the change.

In [ ]:
if experiment_path.exists():
    delta = (after.faithfulness - before.faithfulness).sort_values()
    fig, ax = plt.subplots(figsize=(8, 0.35 * len(delta) + 1))
    for i, qid in enumerate(delta.index):
        ax.plot([before.faithfulness[qid], after.faithfulness[qid]], [i, i], color="#d1d5db")
        ax.scatter(before.faithfulness[qid], i, color=MARK, zorder=3)
        ax.scatter(after.faithfulness[qid], i, color=NEUTRAL, zorder=3)
    ax.set_yticks(range(len(delta)), delta.index, fontsize=8)
    ax.set(xlim=(-0.05, 1.05), xlabel="faithfulness",
           title="Per question: before (black) → after (blue)")  # fmt: skip
    plt.show()
    print("better:", int((delta > 0).sum()), " worse:", int((delta < 0).sum()),
          " same:", int((delta == 0).sum()))  # fmt: skip

## What we learned

1. A score is only meaningful with its **label**: generator, prompt version, judge.
2. **Recompute** headline numbers once by hand; it's the fastest way to understand them.
3. **Averages hide** the few bad answers; histograms and per-question views show them.
4. Faithfulness, relevance and citation validity catch **different** failures.
5. Join retrieval and generation results to find **which layer** to fix.
6. Compare runs only on the **same questions**, and look per question for trade-offs.

**Check yourself**

1. Why is `answer_relevance` averaged over all answers but `faithfulness` only over judged ones?
2. Faithfulness is 0.95 on average. Can the hallucination rate be 40%? How?
3. An answer has faithfulness 1.0 and citation validity 0.5. What happened?
4. Why can't you compare a 10-question experiment with the 30-question baseline directly?